# Pneumonia X-Ray CNN — run this in Google Colab

This notebook clones the repo, installs dependencies, downloads the dataset, trains a
model, and generates the plots/example-prediction image used in the README.

**Before you start:** in the Colab menu, go to `Runtime` → `Change runtime type` →
set **Hardware accelerator** to **GPU**. Training on CPU will be painfully slow.

## 1. Check the GPU is on

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
!git clone https://github.com/rohithsure2000/pneumonia-xray-cnn.git
%cd pneumonia-xray-cnn

## 3. Install dependencies
(Colab already has TensorFlow, but this makes sure versions match `requirements.txt`.)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Upload your Kaggle API token
Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **Create New Token** → this downloads `kaggle.json` to your computer. Run the cell below and choose that file when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select the kaggle.json you just downloaded

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 5. Download the dataset
This pulls the ~1.2 GB Kaggle dataset and unzips it into `data/chest_xray`.

In [ ]:
!bash scripts/download_data.sh data

## 6. Train a model
`improved` is the custom regularized CNN that performed best in the original comparison. Swap `--model` for `basic`, `vgg16`, `vgg19`, or `resnet50` to train a different architecture — run this cell again with a different value to build up the full comparison table.

In [ ]:
!python -m pneumonia_cnn.train \
    --model improved \
    --data-dir data/chest_xray \
    --output-dir artifacts \
    --epochs 15

## 7. Look at the results

In [ ]:
!cat artifacts/improved/test_metrics.json

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/improved/training_curves.png'))

## 8. Generate the example-predictions image for the README
This saves a grid of test images with their true/predicted labels, colored green (correct) or red (incorrect).

In [ ]:
!python scripts/visualize_predictions.py \
    --model-path artifacts/improved/model.keras \
    --data-dir data/chest_xray \
    --output docs/assets/example_predictions.png

In [ ]:
from IPython.display import Image, display
display(Image('docs/assets/example_predictions.png'))

## 9. Download everything back to your computer
This zips up the training curve, metrics, example-predictions image, and the saved model, then downloads it. Unzip it locally and copy `docs/assets/*.png` into your repo's `docs/assets/` folder, then commit and push.

In [ ]:
!zip -r results.zip artifacts docs/assets
from google.colab import files
files.download('results.zip')